# UMKM Sight — Eksplorasi Peramalan Permintaan

Notebook ini dipakai untuk **memahami** pipeline, bukan bagian produksi.
Semua logika yang sama tersedia sebagai modul di `src/umkm_ml/`.

Urutan yang kita ikuti:

1. Lihat data mentah
2. Praproses jadi deret harian
3. Split berbasis waktu
4. Latih & bandingkan 5 model
5. Ramalkan 7 hari
6. Hitung estimasi stok habis

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
from umkm_ml.synthetic_data import generate_sales
from umkm_ml.preprocessing import build_daily_series
from umkm_ml.evaluate import evaluate_models
from umkm_ml.stockout import estimate_stockout

pd.set_option("display.width", 100)
print("import siap")

## 1. Data mentah

Satu baris = satu item transaksi. Bisa banyak baris pada tanggal yang sama,
dan ada baris retur bernilai negatif.

In [ ]:
raw = generate_sales(days=180, seed=42)
print(raw.shape)
raw.head(10)

In [ ]:
raw[raw["quantity"] < 0]

## 2. Praproses

Perhatikan: 180 hari kalender, tapi transaksi hanya terjadi di sebagian hari.
Hari tanpa transaksi **harus muncul sebagai 0**, bukan hilang.

In [ ]:
series, report = build_daily_series(raw, product_id=1)
print(report.to_dict())
print()
print("jumlah hari:", len(series))
series.head(14)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series.index, series.values, linewidth=1)
ax.set_title("Deret harian produk 1 (hari kosong = 0)")
ax.set_ylabel("Kuantitas terjual")
ax.grid(alpha=0.3)
plt.tight_layout()

## 3 & 4. Split berbasis waktu + evaluasi model

Data uji = **28 hari terakhir** (bukan 20% data), karena aplikasi hanya
meramal 7 hari. Rincian akurasi per horizon ada di `horizon_report`.

Semua model memprediksi 28 hari sekaligus; WAPE dipakai sebagai angka
utama karena lebih adil pada data ritel yang banyak hari bernilai kecil.

In [ ]:
result = evaluate_models(series)

print("Latih:", result.train.index.min().date(), "->", result.train.index.max().date(), f"({len(result.train)} hari)")
print("Uji  :", result.test.index.min().date(), "->", result.test.index.max().date(), f"({len(result.test)} hari)")
print("Model terbaik:", result.best_name)

pd.DataFrame([
    {
        "model": s.name,
        "MAE": None if s.mae is None else round(s.mae, 3),
        "RMSE": None if s.rmse is None else round(s.rmse, 3),
        "MAPE %": None if s.mape is None else round(s.mape * 100, 1),
        "WAPE %": None if s.wape is None else round(s.wape * 100, 1),
        "hari nol %": None if s.zero_ratio is None else round(s.zero_ratio * 100, 1),
        "tersedia": s.available,
    }
    for s in result.scores
])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(result.test.index, result.test.values, "k-", label="aktual (uji)", linewidth=2)
for score in result.scores:
    if score.available and score.predictions is not None:
        ax.plot(result.test.index, score.predictions, "--", label=f"{score.name} (MAE {score.mae:.2f})")
ax.set_title("Aktual vs prediksi pada data uji")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()

## 5 & 6. Ramalan 7 hari + estimasi stok habis

Model terbaik dilatih ulang pada **seluruh** data sebelum membuat ramalan,
supaya memakai informasi terbaru.

In [ ]:
from umkm_ml.models import create_model

final_model = create_model(result.best_name).fit(series)
future = final_model.forecast(7)
future_dates = pd.date_range(series.index.max() + pd.Timedelta(days=1), periods=7)

forecast_df = pd.DataFrame({"tanggal": future_dates.date, "prediksi": future.round(2)})
forecast_df

In [ ]:
estimate = estimate_stockout(
    daily_forecast=list(future),
    current_stock=20,
    start_date=future_dates[0].date(),
    lead_time_days=3,
)
estimate.to_dict()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(future_dates[:len(estimate.cumulative_demand)], estimate.cumulative_demand, "o-", label="kumulatif permintaan")
ax.axhline(estimate.current_stock, color="red", linestyle=":", label=f"stok {estimate.current_stock:.0f}")
ax.set_title(estimate.message)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

## Kesimpulan yang bisa dipakai saat presentasi

- ML hanya memprediksi **permintaan harian**; stok habis dihitung dengan aritmatika kumulatif.
- Baseline moving average ikut diuji; model rumit tidak otomatis menang.
- Split data berbasis waktu, bukan acak.
- MAPE dilewati pada hari bernilai 0, dan `zero_ratio` dilaporkan sebagai konteks.
- Data yang belum cukup menghasilkan status `insufficient_data`, bukan angka palsu.